In [1]:
import os

output_folder = "HDB_Output"
os.makedirs(output_folder, exist_ok=True)

print(f"Environment initialized. All datasets will be stored in: '{output_folder}/'")

Environment initialized. All datasets will be stored in: 'HDB_Output/'


OpenAPI Data Extraction

In [9]:
import pandas as pd
import requests

collection_id = 189
url = f"https://api-production.data.gov.sg/v2/public/api/collections/{collection_id}/metadata?withDatasetMetadata=true"

response = requests.get(url)
dataset_list = response.json()['data']['datasetMetadata']

df_list = []
for ds in dataset_list:
    router_url = f"https://api-open.data.gov.sg/v1/public/api/datasets/{ds['datasetId']}/poll-download"
    actual_download_url = requests.get(router_url).json()['data']['url']
    
    # Download straight into Pandas in memory
    df_list.append(pd.read_csv(actual_download_url, dtype=str))

# Unify all eras (1990-2026) and write a single master file to your folder
df_raw = pd.concat(df_list, ignore_index=True)
raw_output_path = os.path.join(output_folder, "dataset_1_raw.csv")
df_raw.to_csv(raw_output_path, index=False)

print(f"Master file downloaded and created at: {raw_output_path}")

Master file downloaded and created at: HDB_Output/dataset_1_raw.csv


Profiling, Field Validation, & Deduplication

In [17]:
# 0. FILE INGESTION (LOAD LOCAL RAW CONSOLIDATED DATASET)
raw_input_path = os.path.join(output_folder, "dataset_1_raw.csv")
df = pd.read_csv(raw_input_path, dtype=str)

print(f"Loaded {df.shape[0]} raw records from local storage.")

# 1. DATA PROFILING & FIELD VALIDATION RULES
valid_month = df['month'].str.match(r'^\d{4}-\d{2}$', na=False)
valid_town = df['town'].str.strip().str.len() > 0
valid_flat_type = df['flat_type'].str.strip().str.len() > 0
valid_flat_model = df['flat_model'].str.strip().str.len() > 0
valid_storey = df['storey_range'].str.match(r'^\d{2} TO \d{2}$', na=False)

passed_profile_mask = valid_month & valid_town & valid_flat_type & valid_flat_model & valid_storey

df_failed_profile = df[~passed_profile_mask].copy()
df_failed_profile['failure_reason'] = "Failed basic field validation/profiling rules"

df_validated = df[passed_profile_mask].copy()
df_validated['resale_price'] = pd.to_numeric(df_validated['resale_price'], errors='coerce')
df_validated = df_validated.dropna(subset=['resale_price'])

# 2. DEDUPLICATION VIA COMPOSITE BUSINESS KEY STRATEGY
df_validated = df_validated.sort_values(by='resale_price', ascending=False)
composite_cols = [col for col in df_validated.columns if col != 'resale_price']
duplicate_mask = df_validated.duplicated(subset=composite_cols, keep='first')

df_failed_duplicates = df_validated[duplicate_mask].copy()
df_failed_duplicates['failure_reason'] = "Duplicate composite key - lower price dropped"

# --- STATE 2: CLEANED BASE ---
df_cleaned = df_validated[~duplicate_mask].copy()

# 3. LEASE CALCULATIONS VIA VECTORIZATION (BASELINE: JULY 2026)
lease_start_years = pd.to_numeric(df_cleaned['lease_commence_date'], errors='coerce')
total_months_lease = 99 * 12
current_year = 2026
current_month = 7 

months_elapsed = ((current_year - lease_start_years) * 12) + (current_month - 1)
months_remaining = (total_months_lease - months_elapsed).clip(lower=0)

remaining_years = months_remaining // 12
remaining_months = months_remaining % 12

# --- STATE 3: TRANSFORMED BASE ---
df_transformed = df_cleaned.copy()
df_transformed['remaining_lease'] = (
    remaining_years.astype(str) + " Years " + remaining_months.astype(str) + " Months"
)

# 4. STATISTICAL OUTLIER DIAGNOSTICS & ANOMALY HEURISTICS
mean_price = df_transformed.groupby('flat_type')['resale_price'].transform('mean')
std_price = df_transformed.groupby('flat_type')['resale_price'].transform('std')
z_scores = (df_transformed['resale_price'] - mean_price) / std_price
df_transformed['is_anomalous'] = np.where(z_scores.abs() > 3, "True", "False")

# --- STATE 4: FAILED BASE ---
failed_dataset_list = [df_failed_profile, df_failed_duplicates]
df_failed = pd.concat([df for df in failed_dataset_list if not df.empty], ignore_index=True)

# 5. INTEGRATE FAILS & CACHE TRANSITIONAL IN-MEMORY BLOCKS
cleaned_master_df = df_cleaned.copy()

print(f"\nProcessing Completed Successfully In-Memory!")

# --- NEW/MODIFIED: STEP 6 - TABLE PREVIEWS ---
print("\n" + "="*40 + " IN-MEMORY TABLE PREVIEWS " + "="*40)
print(f"State 2: CLEANED TABLE PREVIEW (Rows: {df_cleaned.shape[0]}, Cols: {df_cleaned.shape[1]})")
display(df_cleaned[['month', 'town', 'block', 'storey_range', 'resale_price']].head(3))

print(f"\nState 3: TRANSFORMED TABLE PREVIEW (Rows: {df_transformed.shape[0]}, Cols: {df_transformed.shape[1]})")
display(df_transformed[['month', 'town', 'resale_price', 'remaining_lease', 'is_anomalous']].head(3))

print(f"\nState 4: MASTER FAILED TABLE PREVIEW (Rows: {df_failed.shape[0]}, Cols: {df_failed.shape[1]})")
display(df_failed[['month', 'town', 'storey_range', 'resale_price', 'failure_reason']].head(3))
print("="*106)

Loaded 982011 raw records from local storage.

Processing Completed Successfully In-Memory!

======================================== IN-MEMORY TABLE PREVIEWS ========================================
State 2: CLEANED TABLE PREVIEW (Rows: 952664, Cols: 11)


,month,town,block,storey_range,resale_price
224726,2026-04,BUKIT MERAH,96A,46 TO 48,1728000.0
230566,2026-02,QUEENSTOWN,92,19 TO 21,1700000.0
212074,2025-06,QUEENSTOWN,92,22 TO 24,1658888.0



State 3: TRANSFORMED TABLE PREVIEW (Rows: 952664, Cols: 12)


,month,town,resale_price,remaining_lease,is_anomalous
224726,2026-04,BUKIT MERAH,1728000.0,91 Years 6 Months,True
230566,2026-02,QUEENSTOWN,1700000.0,88 Years 6 Months,True
212074,2025-06,QUEENSTOWN,1658888.0,88 Years 6 Months,True



State 4: MASTER FAILED TABLE PREVIEW (Rows: 29347, Cols: 12)


,month,town,storey_range,resale_price,failure_reason
0,2025-12,CLEMENTI,31 TO 33,1480000.0,Duplicate composite key - lower price dropped
1,2026-03,CLEMENTI,22 TO 24,1460000.0,Duplicate composite key - lower price dropped
2,2025-11,CLEMENTI,10 TO 12,1380000.0,Duplicate composite key - lower price dropped


Failure Diagnostics

In [18]:
print("=== BREAKDOWN OF PIPELINE FAILURES ===")
# Uses the unified in-memory failed DataFrame directly
print(df_failed['failure_reason'].value_counts())

print("\n=== SAMPLE OF FAILED RECORDS ===")
# Inspect the rows before they hit the disk
print(df_failed[['month', 'town', 'flat_type', 'storey_range', 'failure_reason']].head())

=== BREAKDOWN OF PIPELINE FAILURES ===
Duplicate composite key - lower price dropped    29347
Name: failure_reason, dtype: int64

=== SAMPLE OF FAILED RECORDS ===
     month      town flat_type storey_range  \
0  2025-12  CLEMENTI    5 ROOM     31 TO 33   
1  2026-03  CLEMENTI    5 ROOM     22 TO 24   
2  2025-11  CLEMENTI    5 ROOM     10 TO 12   
3  2025-08  CLEMENTI    4 ROOM     37 TO 39   
4  2025-07  CLEMENTI    4 ROOM     16 TO 18   

                                  failure_reason  
0  Duplicate composite key - lower price dropped  
1  Duplicate composite key - lower price dropped  
2  Duplicate composite key - lower price dropped  
3  Duplicate composite key - lower price dropped  
4  Duplicate composite key - lower price dropped  


Custom Identifiers & Cryptographic Hashing

In [19]:
# 1. GENERATE THE COMPOSITE STRING FOR HASHING FROM THE STATE 2 (CLEANED) DATASET
composite_string_series = (
    df_cleaned['month'].astype(str) + "_" +
    df_cleaned['town'].astype(str) + "_" +
    df_cleaned['block'].astype(str) + "_" +
    df_cleaned['street_name'].astype(str) + "_" +
    df_cleaned['flat_type'].astype(str) + "_" +
    df_cleaned['storey_range'].astype(str)
)

# --- STATE 5: HASHED BASE ---
df_hashed = df_cleaned.copy()
df_hashed['transaction_hash'] = composite_string_series.apply(
    lambda val: hashlib.sha256(val.encode('utf-8')).hexdigest()
)

# Rearrange columns to place our new primary key hash at the very front position
cols = ['transaction_hash'] + [col for col in df_hashed.columns if col != 'transaction_hash']
df_hashed = df_hashed[cols]

print(f"Cryptographic tracking signatures generated in memory.")

Cryptographic tracking signatures generated in memory.


Export Named Files to Single Directory

In [21]:
# 1. GENERATE THE COMPOSITE STRING FOR HASHING FROM THE STATE 2 (CLEANED) DATASET
composite_string_series = (
    df_cleaned['month'].astype(str) + "_" +
    df_cleaned['town'].astype(str) + "_" +
    df_cleaned['block'].astype(str) + "_" +
    df_cleaned['street_name'].astype(str) + "_" +
    df_cleaned['flat_type'].astype(str) + "_" +
    df_cleaned['storey_range'].astype(str)
)

# --- STATE 5: HASHED BASE ---
df_hashed = df_cleaned.copy()
df_hashed['transaction_hash'] = composite_string_series.apply(
    lambda val: hashlib.sha256(val.encode('utf-8')).hexdigest()
)

# Rearrange columns to place our new primary key hash at the very front position
cols = ['transaction_hash'] + [col for col in df_hashed.columns if col != 'transaction_hash']
df_hashed = df_hashed[cols]

print(f"Cryptographic tracking signatures generated in memory.")

# --- NEW/MODIFIED: STEP 3 - HASHED TABLE PREVIEW ---
print("\n" + "="*40 + " HASHED DATASET AUDIT PREVIEW " + "="*40)
print(f" State 5: HASHED TABLE PREVIEW (Rows: {df_hashed.shape[0]}, Cols: {df_hashed.shape[1]})")
display(df_hashed[['transaction_hash', 'month', 'town', 'block', 'storey_range', 'resale_price']].head(3))
print("="*110)

Cryptographic tracking signatures generated in memory.

======================================== HASHED DATASET AUDIT PREVIEW ========================================
 State 5: HASHED TABLE PREVIEW (Rows: 952664, Cols: 12)


,transaction_hash,month,town,block,storey_range,resale_price
224726,d37d739e36f02b7828e55f80a97d4563ee34c0d8c0c236...,2026-04,BUKIT MERAH,96A,46 TO 48,1728000.0
230566,8b942a7ce087ecaa2acbdcff3b8e839317788cf43fb000...,2026-02,QUEENSTOWN,92,19 TO 21,1700000.0
212074,830a8d2b706ebff244da99b7fb0754339492bfdfe1f292...,2025-06,QUEENSTOWN,92,22 TO 24,1658888.0


Output all files

In [22]:
cleaned_output_path = os.path.join(output_folder, "dataset_1_cleaned.csv")
transformed_output_path = os.path.join(output_folder, "dataset_1_transformed.csv")
failed_output_path = os.path.join(output_folder, "dataset_1_failed.csv")
hashed_output_path = os.path.join(output_folder, "dataset_1_hashed.csv")

# 3. RUN PRODUCTION WRITES
# Exporting the 4 distinct matrices currently cached in memory
df_cleaned.to_csv(cleaned_output_path, index=False)
df_transformed.to_csv(transformed_output_path, index=False)
df_failed.to_csv(failed_output_path, index=False)
df_hashed.to_csv(hashed_output_path, index=False)